# Hồi quy phi tuyến bằng mạng MLP sử dụng PyTorch

Notebook này hướng dẫn bạn cách xây dựng mạng **Multi-Layer Perceptron (MLP)** để giải quyết bài toán **Hồi quy (Regression)** - tức là dự đoán các giá trị số liên tục, thay vì phân loại nhãn.

Chúng ta sẽ tự sinh một tập dữ liệu phi tuyến (hàm hình sin có thêm nhiễu ngẫu nhiên) và dùng mạng MLP để khớp đường cong hàm số đó.

### Sự khác biệt cốt lõi giữa Hồi quy (Regression) và Phân loại (Classification):
1. **Tầng đầu ra (Output Layer)**: 
   * *Classification*: Thường áp dụng hàm kích hoạt phi tuyến ở tầng ra (ví dụ: Sigmoid cho nhị phân, Softmax cho đa lớp) để đưa kết quả dự đoán về dạng xác suất (khoảng `[0, 1]`).
   * *Regression*: **Không sử dụng hàm kích hoạt phi tuyến nào ở tầng ra**. Đầu ra chỉ là một giá trị tuyến tính thô (linear activation), cho phép mô hình dự báo tự do bất kỳ giá trị số thực nào từ $-\infty$ đến $+\infty$.
2. **Hàm mất mát (Loss Function)**:
   * *Classification*: Thường dùng các hàm Entropy chéo như BCE Loss hoặc Cross-Entropy Loss.
   * *Regression*: Dùng các hàm đo lường khoảng cách sai số giữa số thực tế và số dự đoán, phổ biến nhất là **Mean Squared Error (MSE Loss)** hoặc **Mean Absolute Error (MAE Loss)**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập random seed
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {device}")

---
## PHẦN 1: CHUẨN BỊ DỮ LIỆU PHI TUYẾN (HÀM SIN CÓ NHIỄU)

Chúng ta tự sinh một tập dữ liệu dạng sóng sin:
$$y = \sin(x) + \epsilon$$
Trong đó $\epsilon$ là nhiễu ngẫu nhiên có phân phối chuẩn (Gaussian noise). Việc này giúp giả lập dữ liệu thực tế luôn chứa sai số đo lường.

In [ ]:
# 1. Sinh dữ liệu thô
num_samples = 600
X_raw = np.linspace(-3.0, 3.0, num_samples).reshape(-1, 1)
# y = sin(x) + noise
y_raw = np.sin(X_raw) + np.random.normal(0, 0.15, size=X_raw.shape)

# Phân chia tập Train và tập Test (80% train, 20% test)
indices = np.arange(num_samples)
np.random.shuffle(indices)
train_idx, test_idx = indices[:int(0.8*num_samples)], indices[int(0.8*num_samples):]

X_train, y_train = X_raw[train_idx], y_raw[train_idx]
X_test, y_test = X_raw[test_idx], y_raw[test_idx]

# 2. Định nghĩa Custom Dataset cho Regression
class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = RegressionDataset(X_train, y_train)
test_dataset = RegressionDataset(X_test, y_test)

# Khởi tạo DataLoader
batch_size = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# 3. Vẽ dữ liệu để trực quan hóa cấu trúc phi tuyến
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='tab:blue', alpha=0.6, label='Dữ liệu huấn luyện (Train Data)')
plt.plot(np.linspace(-3.0, 3.0, 100), np.sin(np.linspace(-3.0, 3.0, 100)), color='black', linewidth=2.5, label='Hàm chuẩn y = sin(x)')
plt.title("Dữ liệu huấn luyện phi tuyến (Sine Wave with Noise)")
plt.xlabel("Biến độc lập x (Feature)")
plt.ylabel("Biến phụ thuộc y (Target)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## PHẦN 2: XÂY DỰNG MẠNG MLP CHO HỒI QUY

Mạng MLP của chúng ta gồm:
* **Tầng đầu vào**: 1 node (giá trị $x$).
* **Tầng ẩn 1**: 32 nơ-ron + kích hoạt ReLU.
* **Tầng ẩn 2**: 16 nơ-ron + kích hoạt ReLU.
* **Tầng đầu ra**: 1 nơ-ron duy nhất, **không có hàm kích hoạt phi tuyến**.

Hàm mất mát là **MSE Loss** (`nn.MSELoss`) tính toán sai số bình phương trung bình giữa nhãn thực tế và nhãn dự đoán.

In [ ]:
class RegressionMLP(nn.Module):
    def __init__(self, input_dim=1, hidden_dim1=32, hidden_dim2=16, output_dim=1):
        super(RegressionMLP, self).__init__()
        
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        
        out = self.fc2(out)
        out = self.relu(out)
        
        # Tầng đầu ra không dùng activation function
        out = self.fc3(out)
        return out

model = RegressionMLP().to(device)
print(model)

# MSE Loss và Adam Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

---
## PHẦN 3: HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH

Chúng ta thực hiện huấn luyện mô hình trong 200 epochs để mạng MLP có đủ thời gian học cách uốn cong theo sóng sin.

In [ ]:
epochs = 200
loss_history = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for features, targets in train_loader:
        features, targets = features.to(device), targets.to(device)
        
        optimizer.zero_grad()
        predictions = model(features)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * features.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    loss_history.append(epoch_loss)
    
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | MSE Loss: {epoch_loss:.6f}")

# Đánh giá trên tập Test
model.eval()
test_loss = 0.0
with torch.no_grad():
    for features, targets in test_loader:
        features, targets = features.to(device), targets.to(device)
        outputs = model(features)
        loss = criterion(outputs, targets)
        test_loss += loss.item() * features.size(0)

print(f"\nMean Squared Error cuối cùng trên tập Test: {test_loss / len(test_dataset):.6f}")

---
## PHẦN 4: TRỰC QUAN HÓA ĐƯỜNG CONG KHỚP DỮ LIỆU (FITTED CURVE)

Một mô hình hồi quy phi tuyến tốt cần học được quy luật chung của dữ liệu mà không bị overfitting theo nhiễu.

Chúng ta sẽ sinh ra một loạt các điểm x phân bổ đều dọc theo trục hoành, đưa qua mạng MLP dự đoán các giá trị y, rồi vẽ đồ thị để xem đường cong dự báo khớp với dữ liệu thực tế ra sao.

In [ ]:
# 1. Vẽ đồ thị Loss
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='tab:red', label='Train MSE Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Đồ thị MSE Loss qua các epochs huấn luyện')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# 2. Vẽ Fitted Curve dự đoán của mô hình
# Tạo một dãy điểm x phân bổ đều để vẽ đường cong mịn
x_range = np.linspace(-3.0, 3.0, 300).reshape(-1, 1)
x_tensor = torch.tensor(x_range, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    y_predicted = model(x_tensor).cpu().numpy()

plt.figure(figsize=(9, 6))
# Vẽ dữ liệu test thực tế
plt.scatter(X_test, y_test, color='tab:orange', alpha=0.6, label='Dữ liệu kiểm thử thực tế (Test Data)')
# Vẽ đường hàm sin chuẩn (không nhiễu)
plt.plot(x_range, np.sin(x_range), color='black', linestyle='--', linewidth=1.5, label='Hàm chuẩn y = sin(x)')
# Vẽ đường cong mà mạng MLP dự đoán được
plt.plot(x_range, y_predicted, color='red', linewidth=3, label='Đường hồi quy dự đoán bởi MLP (Fitted Curve)')

plt.title("So sánh đường hồi quy dự đoán của mạng MLP với dữ liệu thực tế")
plt.xlabel("Biến độc lập x")
plt.ylabel("Biến dự báo y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()